# 02 - Create the Azure AI Search index

Define the fields, vector profile, query vectorizer, and semantic ranker configuration. This notebook only creates or updates the index; it does not upload documents.

Uses the [Azure vector sample SDK pattern](https://github.com/Azure/azure-search-vector-samples/tree/main/demo-python/code/basic-vector-workflow). Search's managed identity needs **Cognitive Services OpenAI User** on the embedding resource.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField,
    VectorSearch, HnswAlgorithmConfiguration, HnswParameters, VectorSearchProfile,
    AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField,
)

ROOT = Path.cwd() if (Path.cwd() / 'samples').is_dir() else Path.cwd().parent
load_dotenv(ROOT / '.env')
credential = AzureCliCredential(subscription=os.getenv('AZURE_SUBSCRIPTION_ID') or None, process_timeout=60)
index_name = os.getenv('SEARCH_INDEX', 'hr-demo')
dimensions = 3072


## Index fields

`url` is the citation link: the agent shows it on every answer built from the chunk. `metadata_storage_path` holds the same URL because that's the field Copilot Studio's built-in Azure AI Search knowledge source cites. See Citations in the README.

`group_ids` is the [security filter](https://learn.microsoft.com/azure/search/search-security-trimming-for-azure-search) field: a filterable, non-retrievable `Collection(Edm.String)` holding the groups allowed to read each chunk. Manager-only documents carry `managers`; everything else carries `all-employees`. `persona` keeps the source system's own `Manager` / `Non-Manager` value so results can show why a chunk matched — it doesn't enforce anything.

In [ ]:
fields = [
    SimpleField(name='chunk_id', type='Edm.String', key=True, filterable=True),
    SimpleField(name='document_id', type='Edm.String', filterable=True),
    SearchableField(name='title', type='Edm.String'),
    SearchableField(name='section_title', type='Edm.String'),
    SearchableField(name='content', type='Edm.String'),
    SimpleField(name='url', type='Edm.String'),
    SimpleField(name='metadata_storage_path', type='Edm.String'),
]
# Security filter field. Filterable so search.in() can trim on it, and hidden so
# the principal list never comes back in a search response.
fields.append(SimpleField(
    name='group_ids', type='Collection(Edm.String)', filterable=True, hidden=True,
))
# Descriptive metadata, kept retrievable so results show where a chunk came from.
for name in ('persona', 'content_source', 'policy_family', 'revision', 'parent_document_id'):
    fields.append(SimpleField(name=name, type='Edm.String', filterable=True))
fields.append(SearchField(
    name='content_vector', type='Collection(Edm.Single)', searchable=True, hidden=True,
    vector_search_dimensions=dimensions, vector_search_profile_name='hr-vector-profile',
))


In [ ]:
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name='hr-hnsw', parameters=HnswParameters(metric='cosine'))],
    profiles=[VectorSearchProfile(
        name='hr-vector-profile', algorithm_configuration_name='hr-hnsw', vectorizer_name='hr-vectorizer',
    )],
    vectorizers=[AzureOpenAIVectorizer(
        vectorizer_name='hr-vectorizer',
        parameters=AzureOpenAIVectorizerParameters(
            resource_url=os.environ['AZURE_OPENAI_ENDPOINT'],
            deployment_name=os.environ['EMBEDDING_DEPLOYMENT'], model_name='text-embedding-3-large',
        ),
    )],
)
semantic_search = SemanticSearch(
    default_configuration_name='hr-semantic',
    configurations=[SemanticConfiguration(
        name='hr-semantic',
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name='title'), content_fields=[SemanticField(field_name='content')],
        ),
    )],
)
index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search, semantic_search=semantic_search)


## Create the index

This cell calls Azure. For a new corpus/chunking experiment, choose a fresh `SEARCH_INDEX` name in `.env`.

In [ ]:
with SearchIndexClient(endpoint=os.environ['SEARCH_ENDPOINT'], credential=credential) as index_client:
    result = index_client.create_or_update_index(index)
print('Index created:', result.name)
